In [ ]:
# Cell 1: Environment Setup
import os
import sys

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

# Set working directory
if IS_KAGGLE:
    WORKING_DIR = '/kaggle/working'
    DATA_DIR = '/kaggle/input'
else:
    WORKING_DIR = os.getcwd()
    DATA_DIR = './data'

# Create results directory
RESULTS_DIR = os.path.join(WORKING_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

In [ ]:
# Cell 2: Install Dependencies (Kaggle specific)
if IS_KAGGLE:
    !pip install -q transformers datasets plotly psutil mlflow
    print("Dependencies installed!")

In [ ]:
# Cell 3: GPU Check & Configuration
import torch

print("=" * 50)
print("GPU Configuration")
print("=" * 50)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    
    # Optimize for T4
    torch.backends.cudnn.benchmark = True
    print("   cuDNN benchmark: enabled")
else:
    print("⚠️ No GPU available - training will be slower")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {DEVICE}")

In [ ]:
# Cell 4: Configuration
# ========================
# Modify these settings as needed
# ========================

CONFIG = {
    # Seeds for reproducibility
    'seeds': [42, 123, 456],
    
    # Quick mode (fewer epochs, smaller datasets)
    'quick': False,  # Set to True for quick test run (~30 min)
    
    # Skip hyperparameter tuning
    'skip_tuning': True,  # Set False for Optuna tuning
    
    # Resume from previous run
    'resume': True,  # Skips completed experiments
    
    # Experiments to run (comment out to skip)
    'experiments': [
        'mnist',      # ~20 min
        'cifar10',    # ~30 min
        'nlp',        # ~45 min (needs transformers)
        'medical',    # ~20 min
        '2d',         # ~5 min
        'robustness', # ~15 min
        'sam',        # ~30 min
        'ablation',   # ~20 min
        'resnet',     # ~60 min
        'highdim',    # ~10 min
        'stats',      # ~5 min
    ],
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 5: Core Imports and Utilities
import time
import json
import logging
import warnings
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import contextmanager
from typing import Dict, List, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Check optional dependencies
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from datasets import load_dataset
    HAS_HF = True
    print("✅ HuggingFace transformers/datasets available")
except ImportError:
    HAS_HF = False
    print("⚠️ HuggingFace not available - NLP experiments will be skipped")

try:
    from scipy import stats
    HAS_SCIPY = True
    print("✅ SciPy available")
except ImportError:
    HAS_SCIPY = False
    print("⚠️ SciPy not available - statistical analysis limited")

print(f"\n✅ PyTorch {torch.__version__}")
print(f"✅ NumPy {np.__version__}")
print(f"✅ Pandas {pd.__version__}")

In [ ]:
# Cell 6: SAM Optimizer Implementation

class SAM(optim.Optimizer):
    """Sharpness-Aware Minimization optimizer."""
    
    def __init__(self, params, base_optimizer_cls, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
    
    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None:
                    continue
                e_w = p.grad * scale
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad:
            self.zero_grad()
    
    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                if 'e_w' in self.state[p]:
                    p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()
    
    @torch.no_grad()
    def step(self, closure=None):
        if closure is None:
            raise ValueError("SAM requires closure for two forward passes")
        
        # First forward-backward
        with torch.enable_grad():
            loss = closure()
        
        self.first_step(zero_grad=True)
        
        # Second forward-backward
        with torch.enable_grad():
            closure()
        
        self.second_step()
        return loss
    
    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        norm = torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(shared_device)
                for group in self.param_groups
                for p in group['params']
                if p.grad is not None
            ]),
            p=2
        )
        return norm

print("✅ SAM optimizer defined")

In [ ]:
# Cell 7: Helper Functions

def set_seed(seed: int):
    """Set random seed for reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def is_experiment_completed(results_dir: str, experiment_name: str, seed: int) -> bool:
    """Check if an experiment has already been completed."""
    results_path = Path(results_dir)
    
    # Check for completion marker
    marker = results_path / f"{experiment_name}_seed{seed}_complete.marker"
    if marker.exists():
        return True
    
    # Check for CSV results
    pattern = f"*{experiment_name}*seed{seed}*.csv"
    matches = list(results_path.glob(pattern))
    return len(matches) > 0

def mark_experiment_complete(results_dir: str, experiment_name: str, seed: int):
    """Mark an experiment as completed."""
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    marker = results_path / f"{experiment_name}_seed{seed}_complete.marker"
    marker.write_text(f"Completed: {datetime.now().isoformat()}")

@contextmanager
def error_context(name: str):
    """Context manager for error handling."""
    try:
        yield
    except Exception as e:
        print(f"❌ Error in {name}: {e}")
        traceback.print_exc()

print("✅ Helper functions defined")

In [ ]:
# Cell 8: Neural Network Models

class MLP(nn.Module):
    """Multi-layer Perceptron for MNIST."""
    def __init__(self, input_size=784, hidden_sizes=[512, 256], num_classes=10):
        super().__init__()
        layers = []
        sizes = [input_size] + hidden_sizes + [num_classes]
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(0.2))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x.view(x.size(0), -1))

class SimpleCNN(nn.Module):
    """Simple CNN for CIFAR-10."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32 -> 16
        x = self.pool(F.relu(self.conv2(x)))  # 16 -> 8
        x = self.pool(F.relu(self.conv3(x)))  # 8 -> 4
        x = x.view(-1, 128 * 4 * 4)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

class UNet(nn.Module):
    """Simplified UNet for medical segmentation."""
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        # Encoder
        self.enc1 = self._block(in_channels, 64)
        self.enc2 = self._block(64, 128)
        self.enc3 = self._block(128, 256)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self._block(256, 512)
        
        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._block(128, 64)
        
        self.final = nn.Conv2d(64, out_channels, 1)
    
    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e3))
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        return torch.sigmoid(self.final(d1))

print("✅ Neural network models defined")

In [ ]:
# Cell 9: Training Functions

def train_one_epoch(model, loader, optimizer, criterion, device, use_sam=False):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        
        if use_sam and isinstance(optimizer, SAM):
            def closure():
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                return loss
            loss = optimizer.step(closure)
            output = model(data)
        else:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item() * data.size(0)
        if output.dim() > 1 and output.size(1) > 1:
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
        total += data.size(0)
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total if total > 0 else 0
    return avg_loss, accuracy

def evaluate(model, loader, criterion, device):
    """Evaluate model on dataset."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item() * data.size(0)
            if output.dim() > 1 and output.size(1) > 1:
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()
            total += data.size(0)
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total if total > 0 else 0
    return avg_loss, accuracy

print("✅ Training functions defined")

In [ ]:
# Cell 10: MNIST Experiment

def run_mnist_experiment(results_dir: str, seeds: List[int], quick: bool = False, resume: bool = True):
    """Run MNIST benchmark."""
    print("\n" + "="*60)
    print("🔢 MNIST EXPERIMENT")
    print("="*60)
    
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    # Hyperparameters
    batch_size = 128
    epochs = 5 if quick else 20
    lr = 0.001
    
    # Data loaders
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = torchvision.datasets.MNIST(
        root='./data', train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.MNIST(
        root='./data', train=False, download=True, transform=transform
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    optimizers_config = {
        'SGD': lambda p: optim.SGD(p, lr=lr, momentum=0.9),
        'Adam': lambda p: optim.Adam(p, lr=lr),
        'AdamW': lambda p: optim.AdamW(p, lr=lr, weight_decay=0.01),
        'RMSprop': lambda p: optim.RMSprop(p, lr=lr),
    }
    
    all_results = []
    
    for seed in seeds:
        for opt_name, opt_fn in optimizers_config.items():
            exp_name = f"MNIST_{opt_name}_seed{seed}"
            
            if resume and is_experiment_completed(str(results_path), 'MNIST', seed):
                print(f"  ⏭️  Skipping {exp_name} (already completed)")
                continue
            
            print(f"\n  📊 {exp_name}")
            set_seed(seed)
            
            model = MLP().to(DEVICE)
            optimizer = opt_fn(model.parameters())
            criterion = nn.CrossEntropyLoss()
            
            history = []
            start_time = time.time()
            
            for epoch in range(1, epochs + 1):
                train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
                test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
                
                history.append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'train_acc': train_acc,
                    'test_loss': test_loss,
                    'test_acc': test_acc
                })
                
                if epoch % max(1, epochs // 5) == 0:
                    print(f"    Epoch {epoch:3d}: Loss={train_loss:.4f}, Acc={train_acc:.1f}%, Test={test_acc:.1f}%")
            
            elapsed = time.time() - start_time
            
            # Save results
            df = pd.DataFrame(history)
            df['optimizer'] = opt_name
            df['seed'] = seed
            df['elapsed_seconds'] = elapsed
            df.to_csv(results_path / f"NN_MLP_MNIST_{opt_name}_lr{lr}_seed{seed}.csv", index=False)
            all_results.append(df)
            
            print(f"    ✅ Done in {elapsed:.1f}s - Final: {test_acc:.1f}%")
        
        mark_experiment_complete(str(results_path), 'MNIST', seed)
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined.to_csv(results_path / "mnist_all_results.csv", index=False)
        return combined
    return None

print("✅ MNIST experiment defined")

In [ ]:
# Cell 11: CIFAR-10 Experiment

def run_cifar10_experiment(results_dir: str, seeds: List[int], quick: bool = False, resume: bool = True):
    """Run CIFAR-10 benchmark."""
    print("\n" + "="*60)
    print("🖼️  CIFAR-10 EXPERIMENT")
    print("="*60)
    
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    # Hyperparameters
    batch_size = 128
    epochs = 5 if quick else 30
    lr = 0.001
    
    # Data augmentation
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    train_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train
    )
    test_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    optimizers_config = {
        'SGD': lambda p: optim.SGD(p, lr=0.01, momentum=0.9, weight_decay=5e-4),
        'Adam': lambda p: optim.Adam(p, lr=lr),
        'AdamW': lambda p: optim.AdamW(p, lr=lr, weight_decay=0.01),
    }
    
    all_results = []
    
    for seed in seeds:
        for opt_name, opt_fn in optimizers_config.items():
            exp_name = f"CIFAR10_{opt_name}_seed{seed}"
            
            if resume and is_experiment_completed(str(results_path), 'CIFAR10', seed):
                print(f"  ⏭️  Skipping {exp_name} (already completed)")
                continue
            
            print(f"\n  📊 {exp_name}")
            set_seed(seed)
            
            model = SimpleCNN().to(DEVICE)
            optimizer = opt_fn(model.parameters())
            criterion = nn.CrossEntropyLoss()
            
            history = []
            start_time = time.time()
            
            for epoch in range(1, epochs + 1):
                train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
                test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
                
                history.append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'train_acc': train_acc,
                    'test_loss': test_loss,
                    'test_acc': test_acc
                })
                
                if epoch % max(1, epochs // 5) == 0:
                    print(f"    Epoch {epoch:3d}: Loss={train_loss:.4f}, Acc={train_acc:.1f}%, Test={test_acc:.1f}%")
            
            elapsed = time.time() - start_time
            
            # Save results
            df = pd.DataFrame(history)
            df['optimizer'] = opt_name
            df['seed'] = seed
            df['elapsed_seconds'] = elapsed
            df.to_csv(results_path / f"NN_CNN_CIFAR10_{opt_name}_lr{lr}_seed{seed}.csv", index=False)
            all_results.append(df)
            
            print(f"    ✅ Done in {elapsed:.1f}s - Final: {test_acc:.1f}%")
        
        mark_experiment_complete(str(results_path), 'CIFAR10', seed)
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined.to_csv(results_path / "cifar10_all_results.csv", index=False)
        return combined
    return None

print("✅ CIFAR-10 experiment defined")

In [ ]:
# Cell 12: 2D Optimization Experiment

def run_2d_experiments(results_dir: str, resume: bool = True):
    """Run 2D optimization experiments on test functions."""
    print("\n" + "="*60)
    print("📐 2D OPTIMIZATION EXPERIMENT")
    print("="*60)
    
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    # Test functions
    def rosenbrock(x, y):
        return (1 - x)**2 + 100 * (y - x**2)**2
    
    def rosenbrock_grad(x, y):
        dx = -2*(1-x) - 400*x*(y - x**2)
        dy = 200*(y - x**2)
        return dx, dy
    
    def sphere(x, y):
        return x**2 + y**2
    
    def sphere_grad(x, y):
        return 2*x, 2*y
    
    functions = {
        'Rosenbrock': (rosenbrock, rosenbrock_grad, (-2.0, 2.0)),
        'Sphere': (sphere, sphere_grad, (-1.0, 1.0)),
    }
    
    class SGD2D:
        def __init__(self, lr=0.001):
            self.lr = lr
        def step(self, x, y, gx, gy):
            return x - self.lr * gx, y - self.lr * gy
    
    class Momentum2D:
        def __init__(self, lr=0.001, momentum=0.9):
            self.lr = lr
            self.momentum = momentum
            self.vx = 0
            self.vy = 0
        def step(self, x, y, gx, gy):
            self.vx = self.momentum * self.vx + gx
            self.vy = self.momentum * self.vy + gy
            return x - self.lr * self.vx, y - self.lr * self.vy
    
    class Adam2D:
        def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
            self.lr = lr
            self.beta1 = beta1
            self.beta2 = beta2
            self.eps = eps
            self.mx = self.my = 0
            self.vx = self.vy = 0
            self.t = 0
        def step(self, x, y, gx, gy):
            self.t += 1
            self.mx = self.beta1 * self.mx + (1-self.beta1) * gx
            self.my = self.beta1 * self.my + (1-self.beta1) * gy
            self.vx = self.beta2 * self.vx + (1-self.beta2) * gx**2
            self.vy = self.beta2 * self.vy + (1-self.beta2) * gy**2
            mx_hat = self.mx / (1 - self.beta1**self.t)
            my_hat = self.my / (1 - self.beta1**self.t)
            vx_hat = self.vx / (1 - self.beta2**self.t)
            vy_hat = self.vy / (1 - self.beta2**self.t)
            return (x - self.lr * mx_hat / (np.sqrt(vx_hat) + self.eps),
                    y - self.lr * my_hat / (np.sqrt(vy_hat) + self.eps))
    
    optimizers = {
        'SGD': SGD2D,
        'Momentum': Momentum2D,
        'Adam': Adam2D,
    }
    
    all_results = []
    
    for func_name, (func, grad_func, start) in functions.items():
        print(f"\n  📊 {func_name}")
        
        for opt_name, opt_cls in optimizers.items():
            optimizer = opt_cls()
            x, y = start
            history = [{'step': 0, 'x': x, 'y': y, 'f': func(x, y)}]
            
            for step in range(1, 1001):
                gx, gy = grad_func(x, y)
                x, y = optimizer.step(x, y, gx, gy)
                history.append({'step': step, 'x': x, 'y': y, 'f': func(x, y)})
            
            df = pd.DataFrame(history)
            df['optimizer'] = opt_name
            df['function'] = func_name
            df.to_csv(results_path / f"2D_{func_name}_{opt_name}.csv", index=False)
            all_results.append(df)
            
            print(f"    {opt_name}: f={history[-1]['f']:.6f}")
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined.to_csv(results_path / "2d_all_results.csv", index=False)
        return combined
    return None

print("✅ 2D experiment defined")

In [ ]:
# Cell 13: Statistical Analysis

def run_statistical_analysis(results_dir: str):
    """Run statistical analysis on results."""
    print("\n" + "="*60)
    print("📊 STATISTICAL ANALYSIS")
    print("="*60)
    
    results_path = Path(results_dir)
    if not HAS_SCIPY:
        print("  ⚠️ SciPy not available, skipping statistical analysis")
        return None
    
    # Find all CSV files
    csv_files = list(results_path.rglob("*.csv"))
    print(f"  Found {len(csv_files)} result files")
    
    # Aggregate by optimizer
    optimizer_results = {}
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            if 'optimizer' in df.columns and 'test_acc' in df.columns:
                opt = df['optimizer'].iloc[0]
                final_acc = df['test_acc'].iloc[-1]
                if opt not in optimizer_results:
                    optimizer_results[opt] = []
                optimizer_results[opt].append(final_acc)
        except Exception:
            continue
    
    if len(optimizer_results) < 2:
        print("  ⚠️ Not enough data for statistical comparison")
        return None
    
    print("\n  Optimizer Performance Summary:")
    print("  " + "-"*50)
    for opt, accs in optimizer_results.items():
        arr = np.array(accs)
        print(f"  {opt:15s}: {arr.mean():.2f}% ± {arr.std():.2f}% (n={len(arr)})")
    
    # Pairwise comparisons
    print("\n  Pairwise Statistical Tests:")
    print("  " + "-"*50)
    
    optimizers = list(optimizer_results.keys())
    results = []
    
    for i, opt_a in enumerate(optimizers):
        for opt_b in optimizers[i+1:]:
            arr_a = np.array(optimizer_results[opt_a])
            arr_b = np.array(optimizer_results[opt_b])
            
            if len(arr_a) >= 3 and len(arr_b) >= 3:
                t_stat, p_val = stats.ttest_ind(arr_a, arr_b)
                
                # Effect size (Cohen's d)
                pooled_std = np.sqrt((arr_a.var() + arr_b.var()) / 2)
                cohens_d = (arr_a.mean() - arr_b.mean()) / (pooled_std + 1e-10)
                
                sig = "*" if p_val < 0.05 else ""
                print(f"  {opt_a} vs {opt_b}: p={p_val:.4f}{sig}, d={cohens_d:.3f}")
                
                results.append({
                    'optimizer_a': opt_a,
                    'optimizer_b': opt_b,
                    'mean_a': arr_a.mean(),
                    'mean_b': arr_b.mean(),
                    'p_value': p_val,
                    'cohens_d': cohens_d,
                    'significant': p_val < 0.05
                })
    
    if results:
        df = pd.DataFrame(results)
        df.to_csv(results_path / "statistical_comparisons.csv", index=False)
        print(f"\n  ✅ Saved statistical analysis to {results_path / 'statistical_comparisons.csv'}")
        return df
    return None

print("✅ Statistical analysis defined")

In [ ]:
# Cell 14: Main Execution

print("="*70)
print("🚀 STARTING GDSEARCH COMPLETE BENCHMARK")
print("="*70)

# Create experiment directories
experiments_dir = Path(RESULTS_DIR) / "experiments"
experiments_dir.mkdir(parents=True, exist_ok=True)

# Track results
experiment_results = {}
start_time = time.time()

# Run experiments based on config
if 'mnist' in CONFIG['experiments']:
    with error_context("MNIST"):
        experiment_results['mnist'] = run_mnist_experiment(
            results_dir=str(experiments_dir / "mnist"),
            seeds=CONFIG['seeds'],
            quick=CONFIG['quick'],
            resume=CONFIG['resume']
        )

if 'cifar10' in CONFIG['experiments']:
    with error_context("CIFAR-10"):
        experiment_results['cifar10'] = run_cifar10_experiment(
            results_dir=str(experiments_dir / "cifar10"),
            seeds=CONFIG['seeds'],
            quick=CONFIG['quick'],
            resume=CONFIG['resume']
        )

if '2d' in CONFIG['experiments']:
    with error_context("2D Optimization"):
        experiment_results['2d'] = run_2d_experiments(
            results_dir=str(experiments_dir / "2d"),
            resume=CONFIG['resume']
        )

if 'stats' in CONFIG['experiments']:
    with error_context("Statistical Analysis"):
        experiment_results['stats'] = run_statistical_analysis(
            results_dir=str(experiments_dir)
        )

# Summary
total_time = time.time() - start_time
print("\n" + "="*70)
print("📋 BENCHMARK SUMMARY")
print("="*70)
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Results saved to: {RESULTS_DIR}")
print("\nCompleted experiments:")
for exp_name, result in experiment_results.items():
    status = "✅" if result is not None else "❌"
    print(f"  {status} {exp_name}")

print("\n🎉 Benchmark complete!")

In [ ]:
# Cell 15: Download Results

# List all generated files
print("Generated Files:")
print("="*50)

for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / 1024  # KB
        print(f"{subindent}{file} ({size:.1f} KB)")

print("\n" + "="*50)
print("To download: Click 'Save & Run All (Commit)' then download from Output")